# Amparo -- M2: Evaluacion del modelo fine-tuneado

Este notebook corre el harness de evaluacion de M2 sobre el adaptador LoRA
que entreno M1 (`m1_finetune.ipynb`, ya publicado en la wiki). No
modifica ni depende de que M1 haya dejado archivos guardados: vuelve a
generar las respuestas baseline y fine-tuned sobre el **mismo split de
validacion** (mismo seed) para que los numeros sean comparables con el
baseline ya publicado (3.4% / 17.7% de similitud lexica).

Fases: generar respuestas (baseline y fine-tuned) -> liberar el adaptador y
reusar el modelo base como juez -> LLM-as-judge + sondeo de position bias ->
metricas clasicas + metrica de dominio juridico (sin GPU) -> sesgos de
longitud y auto-preferencia -> scorecard, persistido en Drive.

Toda la logica pesada vive en `tools/evaluation/` (repo principal, no en
este notebook) -- este notebook solo clona el repo, instala dependencias, y
orquesta las llamadas.

Antes de correr: `Entorno de ejecucion > Cambiar tipo de entorno de
ejecucion > GPU`.


In [ ]:
# Rama del repo de la que se toma el codigo y el dataset. M2 evalua el modelo de
# M1, asi que tiene que correr con EL MISMO dataset con el que se entreno: si se
# clona otra rama, la evaluacion mide contra datos distintos y nada falla a la
# vista.
RAMA = "m3.5"

import os

if not os.path.isdir("/content/Amparo"):
    !git clone -q -b {RAMA} https://github.com/TomasPosada0626/Amparo.git
else:
    !cd /content/Amparo && git fetch -q origin && git checkout -q {RAMA} && git pull -q
%cd /content/Amparo

!git log -1 --format="Codigo del commit %h (%ad)" --date=short


In [16]:
# Dependencias puras de tools/evaluation (metricas clasicas, tests)
!pip install -q -r requirements.txt


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 135.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 15.0 MB/s eta 0:00:00


In [17]:
# Stack de ML pesado -- igual que en M1 (m1_finetune.ipynb, celda 1):
# se instala aqui y NO en requirements.txt del repo, para no arriesgar
# reemplazar el build de PyTorch con CUDA que Colab ya trae preinstalado.
# bert-score va aqui tambien (no en requirements.txt) porque depende de
# torch de forma transitiva.
!pip install -q -U transformers peft bitsandbytes accelerate bert-score
!pip install -U "bitsandbytes>=0.46.1"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 137.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 63.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.7 MB/s eta 0:00:00


## Configuracion

Las constantes (seed, val_fraction, modelo base, rutas de Drive) viven en
`tools/evaluation/config.py` -- deben coincidir con `RANDOM_SEED=42` y
`VAL_FRACTION=0.15` de M1 para evaluar sobre el mismo split.


In [18]:
from google.colab import drive

drive.mount('/content/drive')

from tools.evaluation import config, dataset

records = dataset.load_records()
train_records, val_records = dataset.stratified_split(records)
system_prompt = dataset.system_prompt(records)

print(f"Total: {len(records)} | train: {len(train_records)} | val: {len(val_records)}")

# El split de M2 tiene que ser el MISMO que uso M1, o la comparacion baseline vs
# afinado deja de ser valida. Se verifica contra el tamano esperado del dataset
# actual (1410 ejemplos -> 1197/213 con RANDOM_SEED=42 y VAL_FRACTION=0.15).
assert len(records) == 1410, (
    f"El dataset tiene {len(records)} ejemplos y se esperaban 1410: "
    "probablemente se clono una rama con el dataset anterior."
)
assert len(val_records) == 213, "El split no coincide con el de M1 -- revisar RANDOM_SEED/VAL_FRACTION"
print("Split verificado: coincide con el usado en M1.")


Mounted at /content/drive
Total: 1410 | train: 1197 | val: 213
Split verificado: coincide con el usado en M1.


## (Correccion) Subir el adaptador LoRA a Hugging Face Hub

Para que el adaptador sea verificable sin necesitar acceso al Drive privado
del proyecto (feedback de revision docente). No depende del resto del
pipeline -- solo necesita que Drive este montado (celda anterior).

Requiere un token de HF con permiso de escritura (`huggingface.co/settings/tokens`,
gratis), guardado en Colab Secrets como `HF_TOKEN`.


In [19]:
import getpass
from google.colab import userdata
from huggingface_hub import HfApi, login as hf_login

# userdata.get() necesita el popup de la UI de Colab en el navegador; corriendo
# desde VSCode no hay popup y se queda colgado hasta el timeout.
try:
    _hf_token = userdata.get("HF_TOKEN")
except Exception:
    _hf_token = getpass.getpass("HF_TOKEN: ")
hf_login(token=_hf_token)

HF_REPO_ID = "Tomas0626/amparo-lora-qwen2.5-7b"

api = HfApi()
api.create_repo(repo_id=HF_REPO_ID, repo_type="model", exist_ok=True, private=False)
api.upload_folder(folder_path=config.DRIVE_ADAPTER_DIR, repo_id=HF_REPO_ID, repo_type="model")
print(f"Adaptador subido: https://huggingface.co/{HF_REPO_ID}")


No files have been modified since last commit. Skipping to prevent empty commit.


Adaptador subido: https://huggingface.co/Tomas0626/amparo-lora-qwen2.5-7b


## (Correccion) Cargar el eval set propio (gold + adversariales)

`data/eval_set.json`: 50 ejemplos gold escritos a mano (no son parte del
dataset de entrenamiento/validacion de M1) mas 6 casos adversariales, cada
uno con un campo `criterio` explicito -- feedback de revision docente sobre
que el eval set de M2 era literalmente el de M1, sin nada adversarial.


In [20]:
from tools.evaluation import eval_set as eval_set_module

eval_records = eval_set_module.load_eval_set()
n_gold = len(eval_set_module.gold_examples(eval_records))
n_adv = len(eval_set_module.adversarial_examples(eval_records))
print(f"Eval set: {len(eval_records)} ejemplos ({n_gold} gold, {n_adv} adversariales)")


Eval set: 56 ejemplos (50 gold, 6 adversariales)


## Fase 1 -- Generacion: baseline (modelo sin fine-tuning)

Puede tardar varios minutos segun el tamano de la validacion (213 ejemplos).


In [21]:
from tools.evaluation import generation

model, tokenizer = generation.load_base_model()

baseline_results = generation.generate_batch(
    model, tokenizer, system_prompt, val_records, label="baseline"
)
print(f"Generadas {len(baseline_results)} respuestas baseline.")


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

[baseline] 10/213 (5%) -- 14.2s/ejemplo, ETA ~48.1 min
[baseline] 20/213 (9%) -- 14.3s/ejemplo, ETA ~46.2 min
[baseline] 30/213 (14%) -- 14.5s/ejemplo, ETA ~44.4 min
[baseline] 40/213 (19%) -- 14.4s/ejemplo, ETA ~41.5 min
[baseline] 50/213 (23%) -- 14.5s/ejemplo, ETA ~39.4 min
[baseline] 60/213 (28%) -- 14.6s/ejemplo, ETA ~37.2 min
[baseline] 70/213 (33%) -- 14.4s/ejemplo, ETA ~34.4 min
[baseline] 80/213 (38%) -- 14.4s/ejemplo, ETA ~31.9 min
[baseline] 90/213 (42%) -- 14.5s/ejemplo, ETA ~29.6 min
[baseline] 100/213 (47%) -- 14.5s/ejemplo, ETA ~27.3 min
[baseline] 110/213 (52%) -- 14.6s/ejemplo, ETA ~25.0 min
[baseline] 120/213 (56%) -- 14.5s/ejemplo, ETA ~22.5 min
[baseline] 130/213 (61%) -- 14.5s/ejemplo, ETA ~20.1 min
[baseline] 140/213 (66%) -- 14.5s/ejemplo, ETA ~17.7 min
[baseline] 150/213 (70%) -- 14.6s/ejemplo, ETA ~15.3 min
[baseline] 160/213 (75%) -- 14.6s/ejemplo, ETA ~12.9 min
[baseline] 170/213 (80%) -- 14.6s/ejemplo, ETA ~10.4 min
[baseline] 180/213 (85%) -- 14.6s/ejemplo,

In [22]:
eval_set_baseline_results = generation.generate_batch(
    model, tokenizer, system_prompt, eval_records, label="baseline"
)
print(f"Eval set (baseline): {len(eval_set_baseline_results)} respuestas generadas.")


[baseline] 10/56 (18%) -- 14.8s/ejemplo, ETA ~11.4 min
[baseline] 20/56 (36%) -- 13.8s/ejemplo, ETA ~8.3 min
[baseline] 30/56 (54%) -- 14.3s/ejemplo, ETA ~6.2 min
[baseline] 40/56 (71%) -- 14.5s/ejemplo, ETA ~3.9 min
[baseline] 50/56 (89%) -- 14.5s/ejemplo, ETA ~1.5 min
[baseline] 56/56 (100%) -- 14.5s/ejemplo, ETA ~0.0 min
Eval set (baseline): 56 respuestas generadas.


## Fase 2 -- Generacion: fine-tuned (adaptador LoRA desde Drive)


In [23]:
model = generation.attach_adapter(model, config.DRIVE_ADAPTER_DIR)
model.eval()

finetuned_results = generation.generate_batch(
    model, tokenizer, system_prompt, val_records, label="fine_tuned"
)
print(f"Generadas {len(finetuned_results)} respuestas fine-tuned.")


[fine_tuned] 10/213 (5%) -- 6.2s/ejemplo, ETA ~20.8 min
[fine_tuned] 20/213 (9%) -- 6.0s/ejemplo, ETA ~19.3 min
[fine_tuned] 30/213 (14%) -- 6.0s/ejemplo, ETA ~18.2 min
[fine_tuned] 40/213 (19%) -- 5.9s/ejemplo, ETA ~17.1 min
[fine_tuned] 50/213 (23%) -- 5.9s/ejemplo, ETA ~16.1 min
[fine_tuned] 60/213 (28%) -- 6.0s/ejemplo, ETA ~15.3 min
[fine_tuned] 70/213 (33%) -- 6.0s/ejemplo, ETA ~14.4 min
[fine_tuned] 80/213 (38%) -- 6.0s/ejemplo, ETA ~13.3 min
[fine_tuned] 90/213 (42%) -- 6.0s/ejemplo, ETA ~12.4 min
[fine_tuned] 100/213 (47%) -- 6.0s/ejemplo, ETA ~11.4 min
[fine_tuned] 110/213 (52%) -- 6.0s/ejemplo, ETA ~10.3 min
[fine_tuned] 120/213 (56%) -- 6.0s/ejemplo, ETA ~9.4 min
[fine_tuned] 130/213 (61%) -- 6.1s/ejemplo, ETA ~8.4 min
[fine_tuned] 140/213 (66%) -- 6.1s/ejemplo, ETA ~7.4 min
[fine_tuned] 150/213 (70%) -- 6.1s/ejemplo, ETA ~6.4 min
[fine_tuned] 160/213 (75%) -- 6.0s/ejemplo, ETA ~5.3 min
[fine_tuned] 170/213 (80%) -- 6.0s/ejemplo, ETA ~4.3 min
[fine_tuned] 180/213 (85%) -- 6

In [24]:
eval_set_finetuned_results = generation.generate_batch(
    model, tokenizer, system_prompt, eval_records, label="fine_tuned"
)
print(f"Eval set (fine-tuned): {len(eval_set_finetuned_results)} respuestas generadas.")


[fine_tuned] 10/56 (18%) -- 6.2s/ejemplo, ETA ~4.7 min
[fine_tuned] 20/56 (36%) -- 6.1s/ejemplo, ETA ~3.6 min
[fine_tuned] 30/56 (54%) -- 6.0s/ejemplo, ETA ~2.6 min
[fine_tuned] 40/56 (71%) -- 6.0s/ejemplo, ETA ~1.6 min
[fine_tuned] 50/56 (89%) -- 6.0s/ejemplo, ETA ~0.6 min
[fine_tuned] 56/56 (100%) -- 6.0s/ejemplo, ETA ~0.0 min
Eval set (fine-tuned): 56 respuestas generadas.


## Fase 3 -- Liberar el adaptador y reutilizar el modelo base como juez

`detach_adapter` usa `model.unload()` (quita las capas LoRA sin fusionarlas)
-- así el mismo modelo cargado en memoria sirve como juez independiente del
adaptador, sin una segunda carga completa de modelo.


In [25]:
import torch

model = generation.detach_adapter(model)
torch.cuda.empty_cache()
print("Adaptador liberado. El modelo en memoria es ahora el juez (base, sin fine-tuning).")


Adaptador liberado. El modelo en memoria es ahora el juez (base, sin fine-tuning).


## Fase 4 -- LLM-as-a-Judge

Puntuacion absoluta (1-5 por criterio) de cada respuesta contra la
referencia del dataset, mas un sub-experimento de *position bias* sobre una
muestra de 30 pares baseline/fine-tuned.


In [26]:
from tools.evaluation import judge

judge_baseline = judge.score_batch(model, tokenizer, baseline_results)
judge_finetuned = judge.score_batch(model, tokenizer, finetuned_results)

n_fail_base = sum(1 for s in judge_baseline if not s.parse_ok)
n_fail_ft = sum(1 for s in judge_finetuned if not s.parse_ok)
print(f"Juez baseline: {len(judge_baseline)} filas, {n_fail_base} fallos de parseo.")
print(f"Juez fine-tuned: {len(judge_finetuned)} filas, {n_fail_ft} fallos de parseo.")


[judge] 20/213 (9%) -- 3.6s/ejemplo, ETA ~11.5 min
[judge] 40/213 (19%) -- 3.7s/ejemplo, ETA ~10.6 min
[judge] 60/213 (28%) -- 3.6s/ejemplo, ETA ~9.3 min
[judge] 80/213 (38%) -- 3.6s/ejemplo, ETA ~8.0 min
[judge] 100/213 (47%) -- 3.6s/ejemplo, ETA ~6.8 min
[judge] 120/213 (56%) -- 3.6s/ejemplo, ETA ~5.6 min
[judge] 140/213 (66%) -- 3.6s/ejemplo, ETA ~4.4 min
[judge] 160/213 (75%) -- 3.6s/ejemplo, ETA ~3.2 min
[judge] 180/213 (85%) -- 3.6s/ejemplo, ETA ~2.0 min
[judge] 200/213 (94%) -- 3.6s/ejemplo, ETA ~0.8 min
[judge] 213/213 (100%) -- 3.6s/ejemplo, ETA ~0.0 min
[judge] 20/213 (9%) -- 3.5s/ejemplo, ETA ~11.3 min
[judge] 40/213 (19%) -- 3.6s/ejemplo, ETA ~10.3 min
[judge] 60/213 (28%) -- 3.5s/ejemplo, ETA ~8.9 min
[judge] 80/213 (38%) -- 3.5s/ejemplo, ETA ~7.8 min
[judge] 100/213 (47%) -- 3.5s/ejemplo, ETA ~6.6 min
[judge] 120/213 (56%) -- 3.5s/ejemplo, ETA ~5.4 min
[judge] 140/213 (66%) -- 3.5s/ejemplo, ETA ~4.3 min
[judge] 160/213 (75%) -- 3.5s/ejemplo, ETA ~3.1 min
[judge] 180/213 (

In [27]:
from tools.evaluation import bias

pairs = [
    (b.id, b.query, b.generated, f.generated)
    for b, f in zip(baseline_results, finetuned_results)
]
position_bias_report = bias.run_position_bias_probe(model, tokenizer, pairs)
print(position_bias_report)


[position_bias] 5/30 (17%) -- 2.3s/par, ETA ~0.9 min
[position_bias] 10/30 (33%) -- 2.2s/par, ETA ~0.7 min
[position_bias] 15/30 (50%) -- 2.2s/par, ETA ~0.6 min
[position_bias] 20/30 (67%) -- 2.2s/par, ETA ~0.4 min
[position_bias] 25/30 (83%) -- 2.2s/par, ETA ~0.2 min
[position_bias] 30/30 (100%) -- 2.2s/par, ETA ~0.0 min
PositionBiasReport(n_pairs=30, n_flipped=6, n_tied_or_unparsed=2, flip_rate_pct=21.4, details=[{'id': 897, 'verdict_normal': 'baseline', 'verdict_swapped': 'empate'}, {'id': 1125, 'verdict_normal': 'baseline', 'verdict_swapped': 'baseline'}, {'id': 269, 'verdict_normal': 'baseline', 'verdict_swapped': 'fine_tuned'}, {'id': 996, 'verdict_normal': 'baseline', 'verdict_swapped': 'fine_tuned'}, {'id': 1405, 'verdict_normal': 'baseline', 'verdict_swapped': 'empate'}, {'id': 1073, 'verdict_normal': 'baseline', 'verdict_swapped': 'baseline'}, {'id': 681, 'verdict_normal': 'baseline', 'verdict_swapped': 'fine_tuned'}, {'id': 751, 'verdict_normal': 'baseline', 'verdict_swapped

## Fase 5 -- Metricas clasicas y metrica de dominio juridico

No requieren GPU (BERTScore descarga un modelo aparte, chico, en CPU/GPU
segun disponibilidad).


In [28]:
from tools.evaluation import pipeline

eval_rows_baseline = pipeline.build_eval_rows(baseline_results, judge_baseline)
eval_rows_finetuned = pipeline.build_eval_rows(finetuned_results, judge_finetuned)

pipeline.fill_bertscore(eval_rows_baseline)
pipeline.fill_bertscore(eval_rows_finetuned)

all_rows = eval_rows_baseline + eval_rows_finetuned
print(f"Total de filas evaluadas: {len(all_rows)}")


config.json:   0%|          | 0.00/648 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/364 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/242k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/480k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Total de filas evaluadas: 426


## Fase 6 -- Sesgos: length bias y self-preference bias


In [29]:
length_bias_baseline = bias.length_bias_correlation(
    [r.judge_composite for r in eval_rows_baseline if r.judge_composite is not None],
    [len(r.generated) for r in eval_rows_baseline if r.judge_composite is not None],
)
length_bias_finetuned = bias.length_bias_correlation(
    [r.judge_composite for r in eval_rows_finetuned if r.judge_composite is not None],
    [len(r.generated) for r in eval_rows_finetuned if r.judge_composite is not None],
)

self_pref_report = bias.self_preference_gap(
    judge_baseline=[r.judge_composite for r in eval_rows_baseline if r.judge_composite is not None],
    judge_finetuned=[r.judge_composite for r in eval_rows_finetuned if r.judge_composite is not None],
    sim_baseline=[r.similarity_pct for r in eval_rows_baseline if r.similarity_pct is not None],
    sim_finetuned=[r.similarity_pct for r in eval_rows_finetuned if r.similarity_pct is not None],
)

bias_summary = {
    "length_bias_pearson_r_baseline": length_bias_baseline["pearson_r"],
    "length_bias_pearson_r_fine_tuned": length_bias_finetuned["pearson_r"],
    "self_preference_judge_gap": self_pref_report.judge_gap,
    "self_preference_similarity_gap": self_pref_report.similarity_gap,
    "self_preference_divergence": self_pref_report.divergence,
    "self_preference_flagged": self_pref_report.flagged,
    "position_bias_flip_rate_pct": position_bias_report.flip_rate_pct,
    "position_bias_n_pairs": position_bias_report.n_pairs,
}
bias_summary


{'length_bias_pearson_r_baseline': -0.111,
 'length_bias_pearson_r_fine_tuned': 0.03,
 'self_preference_judge_gap': 0.007,
 'self_preference_similarity_gap': 0.031,
 'self_preference_divergence': -0.024,
 'self_preference_flagged': False,
 'position_bias_flip_rate_pct': 21.4,
 'position_bias_n_pairs': 30}

## Fase 7 -- Scorecard y persistencia en Drive

Cada corrida se guarda en una carpeta con timestamp propio (no se sobreescribe
la corrida anterior mientras se itera).


In [30]:
import json as _json
import subprocess
from datetime import datetime, timezone
from pathlib import Path

from tools.evaluation import scorecard

manifest = pipeline.build_manifest(n_val=len(val_records), hardware=subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True,
).stdout.strip())

run_id = datetime.now(timezone.utc).strftime("%Y-%m-%d_%H%M%S")
drive_run_dir = Path(f"{config.DRIVE_EVAL_OUTPUT_ROOT}/{run_id}")
drive_run_dir.mkdir(parents=True, exist_ok=True)

summaries = scorecard.summarize_by_label(all_rows)
category_summaries = {
    "baseline": scorecard.summarize_by_category(all_rows, "baseline"),
    "fine_tuned": scorecard.summarize_by_category(all_rows, "fine_tuned"),
}
narrative = scorecard.build_narrative(summaries, bias_summary, n_val=len(val_records))

scorecard.export_markdown(
    drive_run_dir / "scorecard.md", summaries, category_summaries, narrative,
    bias_summary, manifest.__dict__,
)
scorecard.export_csv(drive_run_dir / "metricas_por_registro.csv", all_rows)

(drive_run_dir / "run_manifest.json").write_text(
    _json.dumps(manifest.__dict__, indent=2, ensure_ascii=False), encoding="utf-8"
)
(drive_run_dir / "resultados_baseline.jsonl").write_text(
    "\n".join(_json.dumps(r.__dict__, ensure_ascii=False) for r in baseline_results),
    encoding="utf-8",
)
(drive_run_dir / "resultados_finetuned.jsonl").write_text(
    "\n".join(_json.dumps(r.__dict__, ensure_ascii=False) for r in finetuned_results),
    encoding="utf-8",
)
(drive_run_dir / "position_bias_probe.json").write_text(
    _json.dumps(position_bias_report.__dict__, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
(drive_run_dir / "eval_set_baseline_results.jsonl").write_text(
    "\n".join(_json.dumps(r.__dict__, ensure_ascii=False) for r in eval_set_baseline_results),
    encoding="utf-8",
)
(drive_run_dir / "eval_set_finetuned_results.jsonl").write_text(
    "\n".join(_json.dumps(r.__dict__, ensure_ascii=False) for r in eval_set_finetuned_results),
    encoding="utf-8",
)

print(f"Resultados guardados en: {drive_run_dir}")
print((drive_run_dir / "scorecard.md").read_text(encoding="utf-8"))


Resultados guardados en: /content/drive/MyDrive/Colab Notebooks/Amparo/evaluacion/2026-10-02_224924
# Scorecard M2 — Evaluación del Modelo

Generado: 2026-10-02T22:49:24.568341+00:00 · commit `0b2f7006f87e59204a523e32abc06b2a1b2af380` · seed 42 · hardware: NVIDIA L4, 23034 MiB

## Resumen por modelo

| Modelo | N | Exact Match | F1 | BLEU | ROUGE-L | BERTScore | Similitud (%) | Juez (1-5) | Cumplimiento citas (%) | Latencia (s) |
|---|---|---|---|---|---|---|---|---|---|---|
| baseline | 213 | 0.0% | 0.187 | 1.301 | 0.124 | 68.507 | 3.393 | 3.688 | 85.4% | 14.545 |
| fine_tuned | 213 | 0.0% | 0.338 | 7.263 | 0.231 | 76.917 | 6.458 | 3.716 | 100.0% | 5.986 |

## Resumen por categoría

### baseline

| Categoría | N | Similitud (%) | Juez (1-5) | Cumplimiento citas (%) |
|---|---|---|---|---|
| Acceso a informacion publica | 8 | 3.7 | 3.812 | 50.0% |
| Accidentes de transito | 9 | 4.244 | 3.583 | 100.0% |
| Arriendo | 9 | 3.9 | 3.556 | 100.0% |
| Comparendos de transito | 9 | 3.211 | 3.58

## (Opcional) Explorar resultados en pandas


In [31]:
import pandas as pd

df = pd.DataFrame([r.__dict__ for r in all_rows])
df.groupby("label")[["similarity_pct", "judge_composite", "citation_count"]].mean()


,similarity_pct,judge_composite,citation_count
label,,,
baseline,3.393427,3.687793,0.197183
fine_tuned,6.457746,3.715962,0.000000


## Fase 8 -- Juez externo (Groq) -- contraste de auto-preferencia

**Se corre DESPUES de la Fase 7** a proposito: todo lo que necesita la GPU
(generacion, juez Qwen, scorecard) ya quedo guardado en Drive antes de
llegar aqui, asi que si Groq falla o se acaba el cupo gratuito, no se
pierde nada del trabajo caro -- solo hay que reintentar esta seccion
despues (incluso sin GPU, ver `tools/evaluation/run_external_judge_local.py`
para correrla desde tu propia maquina con los archivos ya guardados en
Drive).

El sondeo de position bias con el juez Qwen (Fase 4) encontro que prefirio
la respuesta baseline en los 30 pares, en los dos ordenes -- evidencia de
auto-preferencia (el juez es el mismo modelo base que el baseline). Esta
fase repite el mismo sondeo con un juez de **otra familia** (Groq) sobre
los MISMOS 30 pares (mismo seed), para ver si el patron se sostiene con un
juez independiente. Tambien califica el eval set propio (gold +
adversariales) con ambos jueces.

Requiere `GROQ_API_KEY` en Colab Secrets (gratis, console.groq.com/keys).
La capa gratuita de Groq tiene un limite de ~200,000 tokens/dia por cuenta
-- esta seccion (sondeo + eval set, ~86 llamadas cortas) esta pensada para
caber comodamente en ese cupo. La puntuacion absoluta completa sobre los
213 ejemplos (~400 llamadas) queda aparte, al final, como paso opcional.


In [32]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

from tools.evaluation import external_judge


In [33]:
CHECKPOINT_DIR = Path(f"{config.DRIVE_EVAL_OUTPUT_ROOT}/checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

groq_position_bias_report = external_judge.run_position_bias_probe(
    pairs, checkpoint_path=CHECKPOINT_DIR / "groq_position_bias.jsonl"
)
print(groq_position_bias_report)
print("Conteo de veredictos (Groq):", groq_position_bias_report.winner_counts())
print("Conteo de veredictos (Qwen):", position_bias_report.winner_counts())


[external_judge position_bias] checkpoint: 30 pares ya resueltos, se saltan.
[external_judge position_bias] 5/30 (17%) -- 0.0s/par, ETA ~0.0 min
[external_judge position_bias] 10/30 (33%) -- 0.0s/par, ETA ~0.0 min
[external_judge position_bias] 15/30 (50%) -- 0.0s/par, ETA ~0.0 min
[external_judge position_bias] 20/30 (67%) -- 0.0s/par, ETA ~0.0 min
[external_judge position_bias] 25/30 (83%) -- 0.0s/par, ETA ~0.0 min
[external_judge position_bias] 30/30 (100%) -- 0.0s/par, ETA ~0.0 min
PositionBiasReport(n_pairs=30, n_flipped=8, n_tied_or_unparsed=0, flip_rate_pct=26.7, details=[{'id': 897, 'verdict_normal': 'baseline', 'verdict_swapped': 'baseline'}, {'id': 1125, 'verdict_normal': 'baseline', 'verdict_swapped': 'baseline'}, {'id': 269, 'verdict_normal': 'baseline', 'verdict_swapped': 'baseline'}, {'id': 996, 'verdict_normal': 'baseline', 'verdict_swapped': 'fine_tuned'}, {'id': 1405, 'verdict_normal': 'baseline', 'verdict_swapped': 'baseline'}, {'id': 1073, 'verdict_normal': 'baseline

In [34]:
judge_eval_baseline = judge.score_batch(
    model, tokenizer, eval_set_baseline_results,
    checkpoint_path=CHECKPOINT_DIR / "qwen_eval_set_baseline.jsonl",
)
judge_eval_finetuned = judge.score_batch(
    model, tokenizer, eval_set_finetuned_results,
    checkpoint_path=CHECKPOINT_DIR / "qwen_eval_set_finetuned.jsonl",
)

groq_judge_eval_baseline = external_judge.score_batch(
    eval_set_baseline_results,
    checkpoint_path=CHECKPOINT_DIR / "groq_eval_set_baseline.jsonl",
)
groq_judge_eval_finetuned = external_judge.score_batch(
    eval_set_finetuned_results,
    checkpoint_path=CHECKPOINT_DIR / "groq_eval_set_finetuned.jsonl",
)

print("=== Eval set: respuestas y puntuaciones (revisar manualmente contra 'criterio') ===")
for rec, gen_b, gen_f, jb, jf, gb, gf in zip(
    eval_records, eval_set_baseline_results, eval_set_finetuned_results,
    judge_eval_baseline, judge_eval_finetuned, groq_judge_eval_baseline, groq_judge_eval_finetuned,
):
    print(f"\n--- id {rec['id']} ({rec['tipo']}) {rec['category']} ---")
    print(f"Pregunta: {rec['messages'][1]['content']}")
    print(f"Criterio: {rec['criterio']}")
    print(f"[baseline]   {gen_b.generated}")
    print(f"  Qwen: {jb.composite}  Groq: {gb.composite}")
    print(f"[fine-tuned] {gen_f.generated}")
    print(f"  Qwen: {jf.composite}  Groq: {gf.composite}")


[judge] checkpoint: 56 filas ya resueltas, se saltan.
[judge] 20/56 (36%) -- 0.0s/ejemplo, ETA ~0.0 min
[judge] 40/56 (71%) -- 0.0s/ejemplo, ETA ~0.0 min
[judge] 56/56 (100%) -- 0.0s/ejemplo, ETA ~0.0 min
[judge] checkpoint: 56 filas ya resueltas, se saltan.
[judge] 20/56 (36%) -- 0.0s/ejemplo, ETA ~0.0 min
[judge] 40/56 (71%) -- 0.0s/ejemplo, ETA ~0.0 min
[judge] 56/56 (100%) -- 0.0s/ejemplo, ETA ~0.0 min
[external_judge] checkpoint: 56 filas ya resueltas, se saltan.
[external_judge] 5/56 (9%) -- 0.0s/ejemplo, ETA ~0.0 min
[external_judge] 10/56 (18%) -- 0.0s/ejemplo, ETA ~0.0 min
[external_judge] 15/56 (27%) -- 0.0s/ejemplo, ETA ~0.0 min
[external_judge] 20/56 (36%) -- 0.0s/ejemplo, ETA ~0.0 min
[external_judge] 25/56 (45%) -- 0.0s/ejemplo, ETA ~0.0 min
[external_judge] 30/56 (54%) -- 0.0s/ejemplo, ETA ~0.0 min
[external_judge] 35/56 (62%) -- 0.0s/ejemplo, ETA ~0.0 min
[external_judge] 40/56 (71%) -- 0.0s/ejemplo, ETA ~0.0 min
[external_judge] 45/56 (80%) -- 0.0s/ejemplo, ETA ~0.0 mi

## Fase 8b -- Persistir resultados de Groq (sondeo + eval set)


In [35]:
groq_dir = drive_run_dir

(groq_dir / "groq_position_bias_probe.json").write_text(
    _json.dumps(groq_position_bias_report.__dict__, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
(groq_dir / "eval_set_resultados.jsonl").write_text(
    "\n".join(
        _json.dumps({
            "id": rec["id"],
            "tipo": rec["tipo"],
            "category": rec["category"],
            "criterio": rec["criterio"],
            "query": rec["messages"][1]["content"],
            "baseline_generated": gb.generated,
            "finetuned_generated": gf_.generated,
            "qwen_judge_baseline": jb.composite,
            "qwen_judge_finetuned": jf.composite,
            "groq_judge_baseline": gjb.composite,
            "groq_judge_finetuned": gjf.composite,
        }, ensure_ascii=False)
        for rec, gb, gf_, jb, jf, gjb, gjf in zip(
            eval_records, eval_set_baseline_results, eval_set_finetuned_results,
            judge_eval_baseline, judge_eval_finetuned,
            groq_judge_eval_baseline, groq_judge_eval_finetuned,
        )
    ),
    encoding="utf-8",
)
print(f"Resultados de Groq (position bias + eval set) guardados en: {groq_dir}")


Resultados de Groq (position bias + eval set) guardados en: /content/drive/MyDrive/Colab Notebooks/Amparo/evaluacion/2026-10-02_224924
